# Plot average income per municipality

Read the **processed** income data and plot a map of Dutch municipalities. This notebook does not change any processed CSV.

**Inputs:** `data/processed/income_per_municipality.csv` and `data/raw/gemeenten_2024.gpkg`

**Output:** `figures/map_income_per_municipality.png`


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
root = Path("..") if Path("../data/raw").exists() else Path(".")


## 1. Load processed income data

In [ ]:
income = pd.read_csv(root / "data/processed/income_per_municipality.csv")
print("Municipalities with income data:", len(income))
display(income.head())


## 2. Join municipal boundaries

In [ ]:
municipalities = gpd.read_file(root / "data/raw/gemeenten_2024.gpkg", layer="gemeente_gegeneraliseerd")

map_name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein"
}
municipalities["municipality"] = municipalities["statnaam"].replace(map_name_mapping)
map_gdf = municipalities.merge(income, on="municipality", how="left")

print("Municipalities without income data:", map_gdf["average_standardized_income_eur"].isna().sum())
print("Income municipalities not on the map:", set(income["municipality"]) - set(municipalities["municipality"]))

## 3. Plot and save map

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))
map_gdf.plot(
    column="average_standardized_income_eur",
    cmap="Reds",
    vmin=34000,
    vmax=62000,
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Average standardized income (€)", "shrink": 0.65},
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax
)
ax.set_title("Average standardized income per municipality (2024)", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
(root / "figures").mkdir(exist_ok=True)
plt.savefig(root / "figures/map_income_per_municipality.png", dpi=200)
plt.show()